In [2]:
# Neural Machine Translation (English → French) using Helsinki-NLP MarianMT

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

from transformers import MarianMTModel, MarianTokenizer, logging
logging.set_verbosity_error()

from datasets import load_dataset
import sacrebleu
import torch

# 1. Load dataset
dataset = load_dataset("opus100", "en-fr", split="test[:20]")

# Prepare data
src_texts = [ex["translation"]["en"] for ex in dataset]
refs = [[ex["translation"]["fr"]] for ex in dataset]

# 2. Load model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Optional: remove warning
model.config.tie_word_embeddings = False

# 3. Tokenize
inputs = tokenizer(src_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)

# 4. Translate
with torch.no_grad():
    translated_tokens = model.generate(**inputs)

translated_texts = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

# 5. Print ONLY translations (clean output)
print("\n--- Translations ---\n")

for i, (src, pred, ref) in enumerate(zip(src_texts, translated_texts, refs), 1):
    print(f"Example {i}")
    print(f"Source    : {src}")
    print(f"Predicted : {pred}")
    print(f"Actual    : {ref[0]}")
    print()

# 6. BLEU score
bleu = sacrebleu.corpus_bleu(translated_texts, refs)
print("BLEU Score:", bleu.score)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]


--- Translations ---

Example 1
Source    : - You were at a bus stop kissing him!
Predicted : - Tu étais à un arrêt de bus pour l'embrasser !
Actual    : - Vous étiez en train de vous embrasser à l'arrêt de bus!

Example 2
Source    : With irony and mischief the young Czech artist Krištof Kintera turns art and life on their heads.
Predicted : Avec l'ironie et le malice, la jeune artiste tchèque Krištof Kintera tourne l'art et la vie sur leur tête.
Actual    : Avec une ironie farceuse, le jeune artiste tchèque Krištof Kintera chamboule l’art et la vie.

Example 3
Source    : - Who's going to talk to the used car salesman?
Predicted : - Qui va parler au vendeur de voitures d'occasion ?
Actual    : Qui va parler au vendeur de voitures ? Big Freddy.

Example 4
Source    : People think you are a great man, that your music speaks of humanity, warmth and understanding.
Predicted : Les gens pensent que vous êtes un grand homme, que votre musique parle de l'humanité, de la chaleur et de la com